# Classification for different Transients using Alerce

## Import libraries

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18
from astropy.time import Time
import sys
import astropy.units as u

In [13]:
from IPython.core.display import HTML
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

And start alerce client

In [14]:
from alerce.core import Alerce
alerce = Alerce()

First let's set up some parameters

In [243]:
#Filter by data from 2026
mjd_min = Time("2026-01-01T00:00:00", scale="utc").mjd
mjd_max = Time("2027-01-01T00:00:00", scale="utc").mjd   
#and the supernovae type
classes_SN = ['SNIa','SNII','SNI','SNIIn','SNIc',]

And query ztf data

In [87]:
df_ztf = alerce.query_objects(
    survey="ztf",
    format="pandas",
    classifier='lc_classifier',
    class_name='SNIa',
    firstmjd=[mjd_min, mjd_max],
    page_size=1000
    )

In [92]:
df_ztf.head()

,oid,ndethist,ncovhist,mjdstarthist,mjdendhist,corrected,stellar,ndet,g_r_max,g_r_max_corr,g_r_mean,g_r_mean_corr,firstmjd,lastmjd,deltajd,meanra,meandec,sigmara,sigmadec,class,classifier,probability,step_id_corr
0,ZTF26aahnjfz,45,4344,61095.226667,61119.171250,True,False,48,None,None,None,None,61095.226667,61119.171250,23.944583,121.838436,15.430817,0.007514,0.007243,SNIa,lc_classifier,0.368480,27.5.7a32.dev1
1,ZTF26aanlofr,26,2224,59250.277153,61146.192836,True,False,25,None,None,None,None,61118.369155,61146.192836,27.823681,128.658593,20.304090,0.008854,0.008304,SNIa,lc_classifier,0.292468,27.5.7a32.dev1
2,ZTF26aactdyg,56,1301,61059.435810,61123.310174,True,False,49,None,None,None,None,61059.435810,61123.310174,63.874363,174.848878,-5.205843,0.003612,0.003597,SNIa,lc_classifier,0.426560,27.5.7a32.dev1
3,ZTF26absfire,129,4158,61286.426030,61327.402627,False,False,128,None,None,None,None,61286.426030,61327.402627,40.976597,29.681919,1.460715,0.002002,0.002001,SNIa,lc_classifier,0.405000,27.6.0
4,ZTF26abqgbkd,57,4550,61275.267859,61301.216146,False,False,57,None,None,None,None,61275.267859,61301.216146,25.948287,312.740374,7.658544,0.002271,0.002251,SNIa,lc_classifier,0.387072,27.6.0


In [244]:
df_alerce = alerce.query_objects(
    survey="lsst",
    format="pandas",
    page_size=10000,
)

In [245]:
df_alerce = df_lsst[df_lsst['class_name']=='SN']

In [246]:
df_alerce.head()

,oid,tid,sid,meanra,meandec,sigmara,sigmadec,firstmjd,lastmjd,deltamjd,n_det,n_forced,n_non_det,stellar,class_name,classifier_name,classifier_version,probability,ranking
0,170591527609303944,1,1,305.582233,-18.790921,0.000003,0.000003,61218.348303,61218.348303,0.000000,1,0,0,None,SN,stamp_classifier_rubin_beta,2.0.1,0.997749,1
1,170591547426865168,1,1,347.099521,-5.778599,0.000003,0.000003,61218.427182,61221.371295,2.944113,2,1,0,None,SN,stamp_classifier_rubin_beta,2.0.1,0.997549,1
2,170591507879297843,1,1,313.427433,-15.723304,0.000002,0.000002,61218.269050,61218.269050,0.000000,1,0,0,None,SN,stamp_classifier_rubin_beta,2.0.1,0.997428,1
3,170591547735146752,1,1,347.009369,-3.854436,0.000005,0.000005,61218.428207,61230.348646,11.920440,2,0,0,None,SN,stamp_classifier_rubin_beta,2.0.1,0.997395,1
4,170648622718779938,1,1,207.309624,-7.442118,0.000002,0.000002,61231.095859,61231.095859,0.000000,1,0,0,None,SN,stamp_classifier_rubin_beta,2.0.1,0.997377,1


Now let's crossmatch this info with Lasair.

The real code start from here

In [15]:
df_lasair = pd.read_csv('Lasair_LSSTuniqueIDs.csv')

In [ ]:
df_lasair.head()

,diaObjectId
0,170019695992242795
1,170019696008495213
2,170019696010067995
3,170019696011640884
4,170019696024748213


In [58]:
import pandas as pd
import pyvo

# Connect to the ALeRCE TAP service
tap_service = pyvo.dal.TAPService("https://tap.alerce.online/tap")

# Build the list of Lasair diaObjectIds for the ADQL IN clause
ids_sql = ",".join(f"'{x}'" for x in df_lasair["diaObjectId"].astype(str))

# Query: objects found in ALeRCE + their most likely class (latest stamp classifier)
query = f"""
SELECT
    obj.oid,
    obj.meanra  AS ra,
    obj.meandec AS dec,
    tax.class_name AS alerce_class
FROM alerce_tap.object AS obj
LEFT JOIN alerce_tap.probability AS prob          -- class probabilities live here
    ON obj.oid = prob.oid AND obj.sid = prob.sid
    AND prob.classifier_id = 3                    -- stamp_classifier_rubin_beta_20260421
    AND prob.ranking = 1                          -- keep only the most likely class
LEFT JOIN alerce_tap.taxonomy AS tax              -- translate class_id -> class name
    ON prob.classifier_id = tax.classifier_id AND prob.class_id = tax.class_id
WHERE obj.sid = 1                                 -- sid = 1: diaObjects (LSST)
  AND obj.oid IN ({ids_sql})
"""

match_alerce = tap_service.search(query, maxrec=1_000_000).to_table().to_pandas()
match_alerce["oid"] = match_alerce["oid"].astype(str)

print(len(match_alerce), "of", len(df_lasair), "found in ALeRCE")
print(match_alerce["alerce_class"].value_counts(dropna=False))
match_alerce.head()

1000 of 1000 found in ALeRCE
alerce_class
bogus       596
AGN         280
SN           86
asteroid     37
VS            1
Name: count, dtype: int64


,oid,ra,dec,alerce_class
0,170019696264873503,60.354730,-50.536077,AGN
1,170019696270114847,61.821563,-48.865792,bogus
2,170019696276406393,60.674362,-49.028122,bogus
3,170019696279552055,60.182272,-49.818652,AGN
4,170019696279552156,59.945907,-49.755205,bogus


In [62]:
match_alerce = match_alerce.drop_duplicates(subset=['oid'])
match_alerce

,oid,ra,dec,alerce_class
0,170019696264873503,60.354730,-50.536077,AGN
1,170019696270114847,61.821563,-48.865792,bogus
2,170019696276406393,60.674362,-49.028122,bogus
3,170019696279552055,60.182272,-49.818652,AGN
4,170019696279552156,59.945907,-49.755205,bogus
...,...,...,...,...
995,170028485678989419,63.905954,-47.917348,bogus
996,170028485693145155,60.875275,-47.697969,bogus
997,170028485696290927,60.919845,-47.106218,AGN
998,170028485698912270,65.582760,-47.308164,bogus


In [64]:
match_alerce.columns

Index(['oid', 'ra', 'dec', 'alerce_class'], dtype='str')

In [61]:
match_alerce.to_csv('match_alerce.csv',index=False)

Now we are going to crossmatch each one with the tns db.


In [ ]:
import lsdb
tns = lsdb.open_catalog("https://data.lsdb.io/hats/tns")
ztf_lsdb = lsdb.from_dataframe(df_ztf, ra_column="meanra", dec_column="meandec")
crossmatch_ztf = ztf_lsdb.crossmatch(tns, radius_arcsec=2, n_neighbors=1,suffix_method='all_columns',
                               suffixes=["_alerce", "_tns"])

In [33]:
results = crossmatch_ztf.compute()

Computing Catalog:   0%|          | 0/8 [00:00<?, ?it/s]

In [39]:
results = results.drop(columns=["reporters_tns"])

In [45]:
results.columns

Index(['oid_alerce', 'ndethist_alerce', 'ncovhist_alerce',
       'mjdstarthist_alerce', 'mjdendhist_alerce', 'corrected_alerce',
       'stellar_alerce', 'ndet_alerce', 'g_r_max_alerce',
       'g_r_max_corr_alerce', 'g_r_mean_alerce', 'g_r_mean_corr_alerce',
       'firstmjd_alerce', 'lastmjd_alerce', 'deltajd_alerce', 'meanra_alerce',
       'meandec_alerce', 'sigmara_alerce', 'sigmadec_alerce', 'class_alerce',
       'classifier_alerce', 'probability_alerce', 'step_id_corr_alerce',
       'objid_tns', 'name_prefix_tns', 'name_tns', 'ra_tns', 'declination_tns',
       'redshift_tns', 'typeid_tns', 'type_tns', 'reporting_groupid_tns',
       'reporting_group_tns', 'source_groupid_tns', 'source_group_tns',
       'discoverydate_tns', 'discoverymag_tns', 'discmagfilter_tns',
       'filter_tns', 'time_received_tns', 'internal_names_tns',
       'Discovery_ADS_bibcode_tns', 'Class_ADS_bibcodes_tns',
       'creationdate_tns', 'lastmodified_tns', '_dist_arcsec'],
      dtype='str')

In [44]:
results.loc[results["oid_alerce"].str.startswith("ZTF17", na=False), ["oid_alerce", "name_tns"]]

,oid_alerce,name_tns
_healpix_29,,
466888586090104656,ZTF17aaaikab,2019bgj
485003778983498981,ZTF17aaaocnx,2019dlf
335874034268710627,ZTF17aaasaat,2016gjt
445318192295641677,ZTF17aaaycpc,2022xms
1605209183325596769,ZTF17aaahrni,2016dxt
1586479779542162972,ZTF17aabbwwl,2026azr


# Antares

In [223]:
from antares_client.search import search as antares_search


In [ ]:
df_lasair["diaObjectId"] = df_lasair["diaObjectId"].astype(str)
ids_lasair = df_lasair["diaObjectId"].tolist()
print(len(ids_lasair), "ids from Lasair")

54 ids de Lasair


In [226]:
def antares_by_ids(ids, chunk_size=100):
    rows = []
    for i in range(0, len(ids), chunk_size):
        chunk = ids[i:i + chunk_size]
        query = {"query": {"bool": {"filter": {
            "terms": {"properties.survey.lsst.dia_object_id": chunk}
        }}}}
        for loc in antares_search(query, limit=100):
            lsst = loc.properties.get("survey", {}).get("lsst", {})
            ids_loc = lsst.get("dia_object_id")
            if not isinstance(ids_loc, list):      # en ANTARES viene como lista
                ids_loc = [ids_loc]
            for oid in ids_loc:                    # una fila por cada id
                rows.append({
                    "diaObjectId": str(oid),
                    "locus_id": loc.locus_id,
                    "ra_antares": loc.ra,
                    "dec_antares": loc.dec,
                    "tags": loc.tags,
                })
    return pd.DataFrame(rows).drop_duplicates(subset="diaObjectId")


def antares_class(tags):
    if not isinstance(tags, list):
        return None
    if "SN_candies" in tags:
        return "SN candidate"
    if "nuclear_transient" in tags:
        return "nuclear"
    if "nova_test" in tags:
        return "nova"
    if "microlensing_candidate" in tags:
        return "microlensing"
    if "extragalactic" in tags:
        return "extragalactic (unclassified)"
    return None

In [228]:
df_antares = antares_by_ids(ids_lasair)
df_antares["antares_class"] = df_antares["tags"].apply(antares_class)

print(len(df_antares), "objets found in ANTARES")
print(df_antares["diaObjectId"].head())
print(df_antares["antares_class"].value_counts(dropna=False))

55 objets found in ANTARES
0    170591514191200614
1    170591522634858650
2    170591507837354141
3    170591507839976058
4    170587109696995855
Name: diaObjectId, dtype: str
antares_class
NaN                             25
SN candidate                    12
extragalactic (unclassified)     6
nova                             6
nuclear                          6
Name: count, dtype: int64


In [232]:
comp = df_lasair.merge(df_antares, on="diaObjectId", how="left")

print(comp["locus_id"].notna().sum(), "from", len(df_lasair), "are in NTARES")

54 from 54 are in NTARES


In [ ]:
match_antares

,diaObjectId,locus_id,ra_antares,dec_antares,tags,antares_class
0,170019717277810735,ANT2026km1wlq37d7g5,151.740280,1.927048,"[refitt_newsources_snrcut, in_LSSTDDF, young_rubin_transients, lsst_scimma_quality_transient, lc_feature_extractor]",NaN
1,170028486191218743,ANT2026fh3qjaltimmx,64.185368,-48.943233,"[lantern_xgboost_t2.0.7_c0.95, in_LSSTDDF, lc_feature_extractor, lsst_scimma_quality_transient]",NaN
2,170028526486945858,ANT2026te116fro3l1u,186.188881,6.589123,[young_rubin_transients],NaN
3,170028526513684557,ANT2026gn80d04elfoz,187.020189,6.967840,"[lsst_scimma_quality_transient, in_shadow_virgo, young_rubin_transients, extragalactic, refitt_newsources_snrcut, lc_feature_extractor]",extragalactic (unclassified)
4,170028526579744791,ANT2026umfmy504a9kc,185.043931,7.798861,[young_rubin_transients],NaN
